# CRIS — PSO-Optimised Retrain
RT-DETR-L retrain from COCO weights using PSO best hyperparameters.

**Two phases:**
- Phase 1 (10 epochs): frozen backbone
- Phase 2 (50 epochs): full fine-tune

**Resume:** if 12h limit hits, download `last.pt` from Output tab, upload as `cluj-retrain-checkpoint` dataset, re-run.

In [ ]:
# Cell 1 — Clone and install
!git clone https://github.com/para0107/Cluj-Road-Intelligence-System.git
%cd Cluj-Road-Intelligence-System
!pip install -r requirements.txt
!pip install loguru openpyxl
!pip uninstall -y wandb ray

In [ ]:
# Cell 2 — Copy datasets
import shutil, os
os.makedirs('data/datasets', exist_ok=True)
print('Copying RDD2022...')
shutil.copytree(
    '/kaggle/input/datasets/paraschiv/cluj-raw-datasets/rdd2022',
    'data/datasets/rdd2022', dirs_exist_ok=True)
print('Copying Pothole600...')
shutil.copytree(
    '/kaggle/input/datasets/paraschiv/cluj-raw-datasets/pothole600',
    'data/datasets/pothole600', dirs_exist_ok=True)
print('Done.')

In [ ]:
# Cell 3 — Flatten nested folders
!mv data/datasets/rdd2022/rdd2022/* data/datasets/rdd2022/ 2>/dev/null || true
!rm -rf data/datasets/rdd2022/rdd2022
!mv data/datasets/pothole600/pothole600/* data/datasets/pothole600/ 2>/dev/null || true
!rm -rf data/datasets/pothole600/pothole600
print('Done.')

In [ ]:
# Cell 4 — Data preparation
!python ml/detection/data_prep/prep_rdd2022.py --dataset_dir data/datasets/rdd2022
!python ml/detection/data_prep/prep_pothole600.py
!python ml/detection/data_prep/merge_datasets.py
!python ml/detection/data_prep/coco_to_yolo.py

In [ ]:
# Cell 5 — Weights + PSO params + resume detection
import json, shutil, os, glob

# COCO pretrained weights — search for any .pt file in the dataset
candidates = glob.glob(
    '/kaggle/input/datasets/paraschiv/cluj-road-weights/**/*.pt*', recursive=True)
print('Weight files found:')
for c in candidates:
    print(f'  {c}')
if not candidates:
    raise FileNotFoundError('No weights found in cluj-road-weights!')
shutil.copy(candidates[0], './rtdetr-l.pt')
print(f'Copied -> rtdetr-l.pt  ({os.path.getsize("rtdetr-l.pt")/1e6:.1f} MB)')

# PSO best hyperparameters
os.makedirs('ml/optimization', exist_ok=True)
pso_best = {
    'lr0':           0.0004465118975867086,
    'weight_decay':  0.000526695253368712,
    'warmup_epochs': 1,
    'mosaic':        0.8603609096800973,
    'mixup':         0.20451311070797243,
    'box':           7.684851652043976,
    'cls':           0.4867776329667799
}
with open('ml/optimization/pso_best.json', 'w') as f:
    json.dump(pso_best, f, indent=2)
print('PSO params written.')

defaults = {'lr0':1e-4,'weight_decay':5e-4,'warmup_epochs':3,
            'mosaic':1.0,'mixup':0.15,'box':7.5,'cls':0.5}
print(f'  {"Param":<20} {"Default":>12}  {"PSO Best":>14}  {"Delta":>8}')
print(f'  {"-"*60}')
for k, v in pso_best.items():
    d = defaults[k]
    print(f'  {k:<20} {str(d):>12}  {v:>14.6f}  {(v-d)/d*100:>+7.1f}%')

# Resume detection
resume_candidates = glob.glob(
    '/kaggle/input/datasets/paraschiv/cluj-retrain-checkpoint/*.pt*', recursive=True)
RESUME_FLAG = ''
if resume_candidates:
    os.makedirs('runs/detect/rtdetr_road/weights', exist_ok=True)
    shutil.copy(resume_candidates[0], 'runs/detect/rtdetr_road/weights/last.pt')
    RESUME_FLAG = '--resume runs/detect/rtdetr_road/weights/last.pt'
    print(f'\nResume checkpoint: {resume_candidates[0]}')
    print(f'RESUME_FLAG = "{RESUME_FLAG}"')
else:
    print('\nNo resume checkpoint — fresh run from COCO weights.')

In [ ]:
# Cell 6 — Background watcher: copies results.csv + last.pt to /kaggle/working/ every 60s
#
# This runs in a daemon thread alongside training.
# Because it is a daemon thread it dies automatically when the notebook kernel stops.
# /kaggle/working/ is the ONLY directory visible in the Kaggle Output tab.
# Every file written there is saved permanently, even if the session dies mid-training.
#
# What gets saved continuously:
#   /kaggle/working/results.csv          <- updated after every epoch
#   /kaggle/working/results.xlsx         <- same data, Excel format
#   /kaggle/working/last.pt              <- latest checkpoint

import threading, shutil, time, os, glob
from pathlib import Path

# Source paths (inside the cloned repo)
RESULTS_CSV_SRC = Path('runs/detect/rtdetr_road/results.csv')
LAST_PT_SRC     = Path('runs/detect/rtdetr_road/weights/last.pt')
BEST_PT_SRC     = Path('runs/detect/rtdetr_road/weights/best.pt')

# Destination — always /kaggle/working/ (visible in Output tab)
OUT = Path('/kaggle/working')

WATCH_INTERVAL = 60   # seconds between checks

def watch_and_copy():
    """Daemon thread: syncs training outputs to /kaggle/working/ every 60 seconds."""
    last_epoch_copied = -1
    while True:
        try:
            # ── results.csv ────────────────────────────────────────────────
            if RESULTS_CSV_SRC.exists():
                import pandas as pd
                df = pd.read_csv(RESULTS_CSV_SRC)
                df.columns = df.columns.str.strip()

                if 'epoch' in df.columns and len(df) > 0:
                    current_epoch = int(df['epoch'].max())

                    if current_epoch > last_epoch_copied:
                        # Add phase column for clarity
                        df.insert(1, 'phase',
                            df['epoch'].apply(
                                lambda e: 'P1-frozen' if e <= 10 else 'P2-full'))

                        # Save CSV
                        df.to_csv(OUT / 'results.csv', index=False)

                        # Save XLSX
                        try:
                            df.to_excel(OUT / 'results.xlsx', index=False)
                        except Exception:
                            pass  # openpyxl may not be available

                        last_epoch_copied = current_epoch
                        print(f'  [watcher] Epoch {current_epoch} synced -> '
                              f'/kaggle/working/results.csv + results.xlsx')

            # ── last.pt ────────────────────────────────────────────────────
            if LAST_PT_SRC.exists():
                shutil.copy2(LAST_PT_SRC, OUT / 'last.pt')

            # ── best.pt ────────────────────────────────────────────────────
            if BEST_PT_SRC.exists():
                shutil.copy2(BEST_PT_SRC, OUT / 'best.pt')

        except Exception as e:
            # Never crash the watcher — just log and continue
            print(f'  [watcher] Warning: {e}')

        time.sleep(WATCH_INTERVAL)

# Start as daemon so it dies cleanly when the kernel stops
watcher = threading.Thread(target=watch_and_copy, daemon=True)
watcher.start()
print(f'Watcher started. Syncing every {WATCH_INTERVAL}s to /kaggle/working/')
print('Files that will always be up to date in the Output tab:')
print('  /kaggle/working/results.csv   <- updated every epoch')
print('  /kaggle/working/results.xlsx  <- same, Excel format')
print('  /kaggle/working/last.pt       <- latest checkpoint')
print('  /kaggle/working/best.pt       <- best checkpoint so far')

In [ ]:
# Cell 7 — Launch training
#
# The watcher (Cell 6) runs in the background and copies results.csv,
# results.xlsx, last.pt, best.pt to /kaggle/working/ every 60 seconds.
# These files are visible in the Output tab and survive session death.
#
# train.py auto-loads ml/optimization/pso_best.json at startup.
# Phase 1: 10 frozen epochs,  lr = 0.000447 (PSO)
# Phase 2: 50 fine-tune epochs, lr = 0.0000447
# SWA over last 5 checkpoints.

import os
cmd = (
    f'python ml/detection/train.py '
    f'--epochs 70 '
    f'--freeze_epochs 10 '
    f'--patience 20 '
    f'--swa_n 5 '
    f'--device 0 '
    f'--workers 4 '
    f'{RESUME_FLAG}'
).strip()
print(f'Command: {cmd}\n')
os.system(cmd)

# ── Auto-save when training finishes ──────────────────────────────────────
# This runs immediately after os.system returns (training complete or crashed).
# It saves swa.pt which the watcher does not copy (only produced at the very end).
import shutil, glob
from pathlib import Path
OUT = Path('/kaggle/working')

for name in ['swa.pt', 'best.pt', 'last.pt']:
    found = glob.glob(f'/kaggle/working/**/weights/{name}', recursive=True)
    if found:
        shutil.copy2(found[0], OUT / name)
        print(f'Saved: {name}')

# Final results.csv copy with phase annotation
import pandas as pd
csvs = glob.glob('/kaggle/working/**/results.csv', recursive=True)
# Exclude the one we already wrote to /kaggle/working/results.csv
src_csvs = [c for c in csvs if 'rtdetr_road' in c]
if src_csvs:
    df = pd.read_csv(src_csvs[0])
    df.columns = df.columns.str.strip()
    df.insert(1, 'phase',
        df['epoch'].apply(lambda e: 'P1-frozen' if e <= 10 else 'P2-full'))
    df.to_csv(OUT / 'results.csv', index=False)
    df.to_excel(OUT / 'results.xlsx', index=False)
    print(f'Final results.csv + results.xlsx saved ({len(df)} epochs)')

print('\nAll outputs in /kaggle/working/ — download from Output tab.')

In [ ]:
# Cell 8 — Print results table (run any time, in a new tab, while training)
import pandas as pd
from pathlib import Path

# Read from /kaggle/working/ — always the latest copy from the watcher
csv_path = Path('/kaggle/working/results.csv')
if not csv_path.exists():
    print('Not available yet — watcher syncs every 60s after epoch 1 completes.')
else:
    df = pd.read_csv(csv_path)
    pd.set_option('display.max_rows', 80)
    pd.set_option('display.width', 160)
    pd.set_option('display.float_format', '{:.5f}'.format)
    print(f'Epochs completed: {int(df["epoch"].max())} / 60')
    display(df)

In [ ]:
# Cell 9 — Plot dashboard (run any time, in a new tab, while training)
import pandas as pd, numpy as np, glob
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

PHASE1_END = 10
BL = {
    'epoch':               [ 1,    3,    5,    7,   10,   11,   19,   20,   38,   39,   56],
    'train/giou_loss':     [1.418,1.157,1.028,0.977,0.942,1.376,0.721,1.047,0.601,0.947,0.568],
    'metrics/recall(B)':   [0.201,0.335,0.447,0.500,0.533,0.084,0.255,0.217,0.348,0.288,0.377],
    'metrics/mAP50(B)':    [0.00248,0.013,0.0204,0.0235,0.0266,0.00761,0.1157,0.0796,0.2107,0.1394,0.273],
    'metrics/mAP50-95(B)': [0.000631,0.00398,0.0069,0.00845,0.00992,None,None,None,None,None,None],
}

# Read from /kaggle/working/ — always the latest watcher copy
csv_path = Path('/kaggle/working/results.csv')
if not csv_path.exists():
    print('Not available yet.')
else:
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    ep_max = int(df['epoch'].max())

    panels = [
        ('train/giou_loss',      'GIoU Loss',  'Train GIoU Loss ↓',        True),
        ('train/cls_loss',       'Cls Loss',   'Train Cls Loss ↓',         True),
        ('train/l1_loss',        'L1 Loss',    'Train L1 Loss ↓',          True),
        ('val/giou_loss',        'Val Loss',   'Val GIoU Loss ↓',          True),
        ('metrics/precision(B)', 'Precision',  'Precision ↑',              False),
        ('metrics/recall(B)',    'Recall',     'Recall ↑',                 False),
        ('metrics/mAP50(B)',     'mAP50',      'mAP50 ↑',                  False),
        ('metrics/mAP50-95(B)', 'mAP50-95',   'mAP50-95 ↑  (PSO target)', False),
    ]

    fig, axes = plt.subplots(2, 4, figsize=(20, 9))
    fig.suptitle(
        f'CRIS RT-DETR-L — PSO Retrain  |  Epoch {ep_max}/60  '
        f'|  Blue = PSO  |  Orange dashed = Baseline',
        fontsize=12, fontweight='bold')

    for ax, (col, ylabel, title, is_loss) in zip(axes.flatten(), panels):
        if col not in df.columns:
            ax.set_title(title + '\n(not yet available)', fontsize=9)
            ax.axis('off')
            continue

        ep  = df['epoch'].values
        val = pd.to_numeric(df[col], errors='coerce').values
        ax.plot(ep, val, color='#1565C0', lw=2, label='PSO retrain', zorder=3)
        ax.scatter(ep, val, color='#1565C0', s=15, zorder=4)

        if col in BL:
            bep  = np.array(BL['epoch'])
            bval = np.array(BL[col], dtype=float)
            mask = ~np.isnan(bval)
            ax.plot(bep[mask], bval[mask], color='#E65100', lw=1.6,
                    linestyle='--', alpha=0.85, label='Baseline', zorder=2)

        ax.axvline(x=PHASE1_END+0.5, color='red', lw=1.2, linestyle=':', alpha=0.7)
        ax.axvspan(0, PHASE1_END+0.5, alpha=0.05, color='grey')

        finite = val[np.isfinite(val)]
        if len(finite):
            best   = finite.min() if is_loss else finite.max()
            best_i = np.where(val == best)[0][0]
            ax.annotate(f'{best:.4f} ep{int(ep[best_i])}',
                        xy=(ep[best_i], best),
                        xytext=(6, 8 if is_loss else -14),
                        textcoords='offset points', fontsize=7, color='#1565C0',
                        arrowprops=dict(arrowstyle='->', color='#1565C0', lw=0.8))

        ax.set_title(title, fontsize=9, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.tick_params(labelsize=7)
        ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True, nbins=8))
        ax.grid(True, alpha=0.3, linestyle=':')
        ax.legend(fontsize=6.5)

    plt.tight_layout()
    plt.savefig('/kaggle/working/training_dashboard.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved -> /kaggle/working/training_dashboard.png')

In [ ]:
# Cell 10 — Evaluate (run after training finishes)
!python ml/detection/evaluate.py --full --compare